# PJM SOUTH DA LMP Forecast — Exploration Notebook

This notebook demonstrates the forecasting pipeline: data generation, feature engineering, model training, and evaluation.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## 1. Mock Data Generation

In [ ]:
from src.data.mock_data import MockDataGenerator

mock = MockDataGenerator(seed=42)
start = '2024-01-01'
end = '2024-12-31'

south_da = mock.generate_pjm_da_lmp(start, end)
print(f'SOUTH DA LMP shape: {south_da.shape}')
print(south_da.head())

print(f'\nPrice stats:')
print(south_da['lmp'].describe())

## 2. Calendar Utils

In [ ]:
from src.data.calendar_utils import CalendarUtils

cal = CalendarUtils()

# Test some dates
test_dates = [
    pd.Timestamp('2024-01-08 10:00'),  # Monday HE10
    pd.Timestamp('2024-01-06 10:00'),  # Saturday HE10
    pd.Timestamp('2024-09-02 10:00'),  # Labor Day HE10
]

for dt in test_dates:
    print(f'{dt}: on-peak={cal.is_onpeak(dt)}, holiday={cal.is_nerc_holiday(dt)}')

## 3. Feature Pipeline

In [ ]:
from src.features.pipeline import FeaturePipeline

pipeline = FeaturePipeline()

features = pipeline.build(
    target_date='2024-06-15',
    whub_onpeak=45.0,
    whub_offpeak=30.0,
    gas_price=3.5,
)

print(f'Feature matrix shape: {features.shape}')
print(f'Columns: {list(features.columns)}')

## 4. Rule-Based Forecast (No Training Required)

In [ ]:
from src.forecast.engine import ForecastEngine

engine = ForecastEngine()

result = engine.forecast(
    target_date='2024-06-15',
    whub_onpeak=45.0,
    whub_offpeak=30.0,
    gas_price=3.5,
)

print(result.to_string(index=False))

## 5. Forecast Plot

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

hours = range(1, 25)
ax.fill_between(hours, result['Lower_90'], result['Upper_90'], alpha=0.3, color='blue', label='90% CI')
ax.plot(hours, result['Forecast_LMP'], 'b-o', linewidth=2, label='Forecast LMP')
ax.plot(hours, result['WHub_DA'], 'g--', linewidth=1.5, label='WHub DA')

# Shade on-peak hours
onpeak = result['Is_OnPeak'] == 'On-Peak'
for i, is_op in enumerate(onpeak, 1):
    if is_op:
        ax.axvspan(i - 0.5, i + 0.5, alpha=0.1, color='yellow')

ax.set_xlabel('Hour Ending (EPT)')
ax.set_ylabel('LMP ($/MWh)')
ax.set_title('PJM SOUTH DA LMP Forecast — 2024-06-15')
ax.legend()
ax.set_xticks(range(1, 25))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()